# Vision 실습 문제 (Practice)

각 셀에서 `## Write Answer` 부분을 채워보세요.

## 출제 범위
- **데이터 구성 & Library 활용**: torchvision transforms로 학습/평가 데이터 파이프라인 구성
- **평가 Metric**: Top-1 Accuracy, Confusion Matrix 계산 방법
- **모델 원리(ResNet/ViT/DETR/U-Net)**: 주요 레이어 구성, 동작 원리, Skip/Attention/Pooling 등
- **학습 원리**: Optimizer, 학습 루프 3단계(zero_grad/backward/step)

## 문제 1. CIFAR-10 학습용 Transform 구성

CIFAR-10 학습 파이프라인에 `RandomCrop(32, padding=4)`, `RandomHorizontalFlip`, `ToTensor`, `Normalize`를 순서대로 구성하세요.

In [ ]:
import torchvision.transforms as T

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

train_tfms = T.Compose([
    T.## Write Answer(32, padding=4),
    T.## Write Answer(),
    T.## Write Answer(),
    T.## Write Answer(CIFAR10_MEAN, CIFAR10_STD),
])

## 문제 2. Top-1 Accuracy 계산

`logits` (B, num_classes)과 `targets` (B,)로부터 Top-1 Accuracy를 반환하는 코드를 완성하세요.

In [ ]:
def accuracy_top1(logits, targets):
    preds = ## Write Answer
    return (preds == targets).float().mean().item()

## 문제 3. Confusion Matrix 집계

정답(`labels`)과 예측(`preds`)을 이용해 `cm[GT, Pred]` 카운트를 1 증가시키도록 인덱스를 채우세요.

In [ ]:
import torch

def confusion_and_perclass(preds, labels, num_classes):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for t, p in zip(labels, preds):
        cm[## Write Answer, ## Write Answer] += 1
    per_class_acc = cm.diagonal().float() / torch.clamp(cm.sum(dim=1).float(), min=1.0)
    return cm, per_class_acc

## 문제 4. 학습 루프의 핵심 3단계

PyTorch 학습 루프의 **zero_grad → backward → step** 순서를 완성하세요.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        ## Write Answer
        logits = model(images)
        loss = criterion(logits, labels)
        ## Write Answer
        ## Write Answer

## 문제 5. SGD + Momentum Optimizer

ResNet18 학습에 사용할 `SGD`를 `lr=0.1`, `momentum=0.9`, `weight_decay=5e-4`로 생성하세요.

In [ ]:
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = ## Write Answer

## 문제 6. ViT PreNorm (LayerNorm)

ViT Encoder의 Pre-LN 패턴에서 토큰 임베딩 차원(`dim`) 기준 LayerNorm 레이어를 선언하세요.

In [ ]:
import torch.nn as nn

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = ## Write Answer
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

## 문제 7. ViT FeedForward (Linear → GELU → Linear)

Transformer FFN 블록을 `Linear(dim→hidden) → GELU → Dropout → Linear(hidden→dim) → Dropout` 구조로 완성하세요.

In [ ]:
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.## Write Answer,
            nn.## Write Answer,
            nn.Dropout(dropout),
            nn.## Write Answer,
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

## 문제 8. Multi-Head Attention inner_dim 계산

Multi-Head Attention에서 전체 헤드 차원 `inner_dim`을 `heads`와 `dim_head`로 계산하는 식을 완성하세요.

In [ ]:
import torch.nn as nn

class AttentionInit(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        inner_dim = ## Write Answer
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

## 문제 9. Scaled Dot-Product Attention (einsum)

(1) Q와 K의 내적으로 `dots` 점수를 `einsum`으로 계산하고 `self.scale`을 곱하세요. (2) `attn`은 `self.attend`(softmax)로 확률화하세요.

In [ ]:
import torch
from torch import nn, einsum
from einops import rearrange

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)

        dots = ## Write Answer
        attn = ## Write Answer

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

## 문제 10. Transformer Block 구성

한 Transformer 층은 `(PreNorm + Attention)`, `(PreNorm + FeedForward)` 두 서브블록으로 구성됩니다. 빈칸을 채워 각 서브블록을 완성하세요.

In [ ]:
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                ## Write Answer,
                ## Write Answer,
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x

## 문제 11. ViT 패치 임베딩 (einops Rearrange 패턴)

입력 이미지 `(B, C, H, W)`를 `P×P` 패치 토큰 시퀀스 `(B, N, P·P·C)`로 바꾸는 `Rearrange` 패턴 문자열을 완성하세요.

In [ ]:
from einops.layers.torch import Rearrange
import torch.nn as nn

patch_height = 4
patch_width = 4
patch_dim = 3 * patch_height * patch_width
dim = 64

to_patch_embedding = nn.Sequential(
    Rearrange(## Write Answer, p1=patch_height, p2=patch_width),
    nn.Linear(patch_dim, dim),
)

## 문제 12. ViT CLS 토큰 결합 & 위치 임베딩

(1) `cls_token`을 배치 크기 `b`만큼 복제하는 einops `repeat` 패턴 문자열을 채우고, (2) 시퀀스 맨 앞에 concat하며, (3) 길이 `n+1`만큼 위치 임베딩을 더하세요.

In [ ]:
import torch
from torch import nn
from einops import repeat

class ViTForward(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))

    def forward(self, x):
        b, n, _ = x.shape
        cls_tokens = repeat(self.cls_token, ## Write Answer, b=b)
        x = torch.cat(## Write Answer)
        x = x + self.pos_embedding[:, :(## Write Answer)]
        return x

## 문제 13. ViT 분류 표현 추출: CLS 토큰 vs mean pooling

`pool == "cls"`이면 CLS 토큰 벡터를, `pool == "mean"`이면 토큰 평균 풀링 벡터를 반환하세요.

In [ ]:
def select_feature(x, pool):
    x = ## Write Answer
    return x

## 문제 14. DETR 입력 전처리 파이프라인

DETR 공식 데모에 맞춰 **짧은 변을 800픽셀로 Resize → ToTensor → ImageNet mean/std Normalize** 순서로 구성하세요.

In [ ]:
import torchvision.transforms as T

transform = T.Compose([
    T.## Write Answer(800),
    T.## Write Answer(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 문제 15. 박스 좌표 변환 (cx,cy,w,h) → (xmin,ymin,xmax,ymax)

박스 성분에서 좌상·우하 꼭짓점 4개를 계산한 리스트 `b`를 `(N, 4)` 모양의 텐서로 다시 합치세요.

In [ ]:
import torch

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return ## Write Answer

## 문제 16. 정규화 박스를 픽셀 좌표로 복원

DETR 출력은 `(0~1)`로 정규화된 `(xmin,ymin,xmax,ymax)` 좌표이므로, PIL 이미지 `size=(W,H)`를 이용해 `(W, H, W, H)` 스케일로 곱해 픽셀 단위로 복원하세요.

In [ ]:
import torch

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor(## Write Answer, dtype=torch.float32)
    return b

## 문제 17. DETR 예측 신뢰도 필터링

`outputs['pred_logits']` shape = `(1, num_queries, num_classes+1)`에서 마지막 클래스는 *no-object*입니다. (1) 클래스 확률로 변환하고 (2) no-object를 제외한 뒤 (3) 각 query의 최대 score가 0.9를 넘는 것만 `keep`에 True로 남기세요.

In [ ]:
probas = outputs['pred_logits'].softmax(## Write Answer)[0, :, :## Write Answer]
keep = ## Write Answer

## 문제 18. DETR keep 마스크로 박스 선택

`outputs['pred_boxes']` 첫 번째 배치에서 `keep==True`인 쿼리만 골라 `rescale_bboxes`에 전달해 픽셀 좌표 박스를 얻으세요.

In [ ]:
bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, ## Write Answer], im.size)

## 문제 19. U-Net DoubleConv 블록

`Conv3x3(padding=1) → ReLU → Conv3x3(padding=1) → ReLU`로 구성된 U-Net 기본 블록 내부를 완성하세요. `bias=False`를 사용합니다.

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.net = nn.Sequential(
            ## Write Answer,
            ## Write Answer,
            ## Write Answer,
            ## Write Answer,
        )

    def forward(self, x):
        return self.net(x)

## 문제 20. U-Net Down 블록 (MaxPool)

Downsampling 블록에서 공간 해상도를 절반으로 줄이는 **MaxPool2d(kernel_size=2)**를 선언하세요.

In [ ]:
import torch.nn as nn

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = ## Write Answer
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)
        return x

## 문제 21. U-Net Up 블록 (Upsample / ConvTranspose2d)

(1) `bilinear=True`이면 파라미터가 없는 **Upsample(scale_factor=2, mode='bilinear', align_corners=True)**을, (2) `bilinear=False`이면 학습 가능한 **ConvTranspose2d**(채널 절반, kernel=2, stride=2)를 사용하세요.

In [ ]:
import torch.nn as nn

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=False):
        super().__init__()
        if bilinear:
            self.up = ## Write Answer
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2)
        else:
            self.up = ## Write Answer
            self.conv = DoubleConv(in_channels, out_channels)

## 문제 22. U-Net Skip Connection (채널 방향 concat)

업샘플된 디코더 feature `x1`과 인코더 skip feature `x2`를 **채널(dim=1) 축**으로 결합하세요.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UpForward(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                        diff_y // 2, diff_y - diff_y // 2])
        x = ## Write Answer
        return self.conv(x)

## 문제 23. U-Net 전체 구조 (입력 stem & 디코더 Skip Connection 호출)

(1) `self.inc`의 입력 채널은 `n_channels`로 지정하세요. (2) `forward`에서 각 `up` 블록에 **디코더 feature(저해상도)**와 **인코더 skip(고해상도)**을 올바른 순서로 전달하세요.

In [ ]:
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super().__init__()
        self.inc = DoubleConv(## Write Answer, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(## Write Answer, ## Write Answer)
        x = self.up2(## Write Answer, ## Write Answer)
        x = self.up3(## Write Answer, ## Write Answer)
        x = self.up4(## Write Answer, ## Write Answer)
        return self.outc(x)